# PneumoniaMNIST – Klasyfikacja zdjęć rentgenowskich płuc w kierunku zapalenia płuc
**Przedmiot:** Uczenie Maszynowe w Diagnostyce Medycznej (Politechnika Krakowska)  
**Zespół:** Kamil Rayski, Miłosz Pabis  
**Zbiór danych:** [Kaggle: Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia) / PneumoniaMNIST ($n = 5\,856$ zdjęć RTG)

*Notatnik wygenerowany automatycznie (`build_notebook.py`, 2026-11-15 22:45 UTC) na podstawie bieżących artefaktów w `models/` i `figures/`.*

---
## 1. Wstęp i pobranie danych (`kagglehub`)
Pełny potok trenowania i ewaluacji: `uv run python run_pipeline.py` (grupowy podział walidacyjny, potoki sklearn, CNN, bootstrap na teście).


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPyImage, display

from config import (
    DATA_DIR,
    DATASET_METADATA_PATH,
    FIGURES_DIR,
    KAGGLE_DATASET_ID,
    METRICS_SUMMARY_PATH,
)

os.environ["KAGGLEHUB_CACHE"] = str(DATA_DIR)

# Opcjonalnie: pobranie zbioru (pomijane, jeśli cache już istnieje)
try:
    import kagglehub
    path = kagglehub.dataset_download(KAGGLE_DATASET_ID)
    print("Path to dataset files:", path)
except Exception as exc:
    print("kagglehub download skipped:", exc)


## 2. Eksploracyjna Analiza Danych (EDA)
Wczytujemy metadane $5\,856$ zdjęć RTG. Walidacja $n=524$ pochodzi ze **stratyfikowanego, grupowego** podziału
(`StratifiedGroupKFold`) — ten sam pacjent/studium nie występuje w `train` i `val`.


In [ ]:
meta_df = pd.read_csv(DATASET_METADATA_PATH)
print(f"Łączna liczba zdjęć RTG: {len(meta_df)}")
print("\n--- Oryginalny podział (Kaggle) ---")
display(pd.crosstab(meta_df["orig_split"], meta_df["label_name"], margins=True))
print("\n--- Skorygowany podział (PneumoniaMNIST + grupy pacjent/studium) ---")
display(pd.crosstab(meta_df["split"], meta_df["label_name"], margins=True))
if "study_group" in meta_df.columns:
    print("\nLiczba unikalnych grup study_group per split:")
    display(meta_df.groupby("split")["study_group"].nunique().to_frame("Liczba grup"))
print("\n--- Podział etiologiczny ---")
display(meta_df["subtype"].value_counts().to_frame("Liczba zdjęć"))

with open(METRICS_SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary = json.load(f)
display(pd.DataFrame([summary.get("leakage_audit", {})]).T)


In [ ]:
for fig in ("fig1_class_distribution.png", "fig2_sample_xray_grid.png", "fig3_image_stats_eda.png"):
    display(IPyImage(filename=str(FIGURES_DIR / fig)))


## 3. Przetwarzanie danych, Inżynieria Cech i Redukcja Wymiarowości (PCA)
Klasyczne modele: $64\times 64$ px + 17 cech domenowych + PCA ($k=100$) wewnątrz potoku sklearn.
Sieć CNN: $128\times 128$ px, augmentacja geometryczna, ważona `CrossEntropyLoss`.


In [ ]:
display(IPyImage(filename=str(FIGURES_DIR / 'fig4_pca_analysis.png')))


## 4. Modelowanie, Strojenie Hiperparametrów i Ewaluacja

Porównano cztery modele (preprocessing w potokach `sklearn`, **5-fold StratifiedGroupKFold** po identyfikatorze studium/pacjenta — bez wycieku między partycjami). Progi decyzyjne dobierano wyłącznie na walidacji (Youden J); na test nanoszono je bez strojenia. Prawdopodobieństwa raportowano po skalowaniu **Platta** (fit na walidacji).

- PCA ($k=100$) na zbiorze treningowym wyjaśnia **91.52%** wariancji.

**Wybrane konfiguracje (z `metrics_summary.json`):**

- **Logistic Regression (PCA)** — CV ROC-AUC = `0.9916`, parametry: `C=1.0`, `class_weight='balanced'`
- **Random Forest** — CV ROC-AUC = `0.9872`, parametry: `class_weight='balanced_subsample'`, `max_depth=None`, `n_estimators=150`
- **SVM (RBF + PCA)** — CV ROC-AUC = `0.9951`, parametry: `C=5.0`, `class_weight=None`, `gamma='scale'`
- **PneumoniaCNN (PyTorch)** — CV ROC-AUC = `0.9977`, parametry: `img_size='128x128'`, `optimizer='AdamW (lr=1e-3, wd=1e-4)'`, `scheduler='CosineAnnealingLR'`, `loss='Weighted CrossEntropy'`, `epochs=15`

**Audyt rozłączności grup:** liczba grup train/val/test = {'train': 3304, 'val': 366, 'test': 428}; wszystkie przecięcia = 0 obrazów.


In [ ]:
from report_metrics import (
    build_hyperparams_dataframe,
    build_paired_bootstrap_dataframe,
    build_results_dataframe,
    build_thresholds_dataframe,
)

with open(METRICS_SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary = json.load(f)

print("=== Hiperparametry ===")
display(build_hyperparams_dataframe(summary))
print("=== Progi walidacyjne (raw + Platt) ===")
display(build_thresholds_dataframe(summary))
print("=== Metryki (walidacja / test @0.5 / test @próg z walidacji) ===")
display(build_results_dataframe(summary))
pairs = build_paired_bootstrap_dataframe(summary)
if len(pairs):
    print("=== Paired bootstrap ROC-AUC (test) ===")
    display(pairs)


In [ ]:
for fig in (
    "fig7_cnn_training_curves.png",
    "fig5_model_comparison_roc_pr.png",
    "fig6_confusion_matrices.png",
):
    display(IPyImage(filename=str(FIGURES_DIR / fig)))


## 5. Wyjaśnialność Modeli (XAI) oraz Analiza Błędów
Mapy istotności / Grad-CAM oraz analiza FP/FN CNN (próg Platt z walidacji).


In [ ]:
with open(METRICS_SUMMARY_PATH, "r", encoding="utf-8") as f:
    summary = json.load(f)
display(IPyImage(filename=str(FIGURES_DIR / "fig8_explainability_gradcam_importance.png")))
display(IPyImage(filename=str(FIGURES_DIR / "fig9_error_analysis.png")))
if "subtype_test_accuracy_cnn" in summary:
    display(pd.DataFrame(summary["subtype_test_accuracy_cnn"]).T)


## 6. Implementacja – Streamlit (`app.py`)
Skalibrowane P(Platt), osobne progi walidacyjne dla każdego modelu.

```bash
uv run streamlit run app.py
```


In [ ]:
display(IPyImage(filename=str(FIGURES_DIR / 'fig10_ui_preview.png')))
